In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# --- Base project paths (Google Drive) ---
PROJECT = "player_detection"
BASE = Path("/content/drive/My Drive/DS&AI/Projects") / PROJECT
VID_DIR = BASE / "videos"
FRAME_DIR = BASE / "data" / "images_all"
DATA_DIR = BASE / "data"
TRAIN_IMG = DATA_DIR / "images" / "train"
VAL_IMG = DATA_DIR / "images" / "val"
TRAIN_LAB = DATA_DIR / "labels" / "train"
VAL_LAB = DATA_DIR / "labels" / "val"
EXPORTS = BASE / "runs"

# Create directories if they don’t exist
for p in [VID_DIR, FRAME_DIR, TRAIN_IMG, VAL_IMG, TRAIN_LAB, VAL_LAB, EXPORTS]:
    p.mkdir(parents=True, exist_ok=True)

print("✅ Project folder:", VID_DIR)

NameError: name 'Path' is not defined

In [ ]:
!pip -q install --upgrade ultralytics yt-dlp lap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 92.5 MB/s eta 0:00:00


In [ ]:
import os
import glob
import random
import shutil
import subprocess
from pathlib import Path

import torch
from ultralytics import YOLO

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.11.0+cu128
CUDA available: True


In [ ]:
VIDEOS = [
    "https://www.youtube.com/shorts/T9m3ACzBTA4",
    "https://www.youtube.com/shorts/B-2E53Rxt4U",
    "https://www.youtube.com/shorts/iNVvvhsjdvM",
    "https://www.youtube.com/shorts/bmAUr22ap8c",
    "https://www.youtube.com/shorts/y02mOQudT3E",
    "https://www.youtube.com/shorts/c-djMIii9jQ",
    "https://www.youtube.com/shorts/qGWjm5K6Ux8",
    "https://www.youtube.com/watch?v=4AwYxLFcX74",
    "https://www.youtube.com/shorts/hIsYsd4yX7o",
    "https://www.youtube.com/shorts/dqQX3URGRCo"
]

CLIP_SECONDS = 8
TARGET_WIDTH = 1280
FPS = 10
VAL_SPLIT = 0.2
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

print("Number of videos:", len(VIDEOS))

Number of videos: 10


In [ ]:
from yt_dlp import YoutubeDL

def download_videos(urls, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)

    ydl_opts = {
        "outtmpl": str(out_dir / "%(id)s.%(ext)s"),
        "format": "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
        "merge_output_format": "mp4",
        "noplaylist": True,
        "quiet": True,
    }

    downloaded = []

    with YoutubeDL(ydl_opts) as ydl:
        for url in urls:
            try:
                info = ydl.extract_info(url, download=True)
                video_id = info.get("id")

                path = next(
                    (p for p in out_dir.glob(f"{video_id}.*") if p.suffix in [".mp4", ".mkv", ".webm"]),
                    None
                )

                if path:
                    downloaded.append(path)
                    print("Downloaded:", path.name)

            except Exception as e:
                print("Download failed:", url)
                print(e)

    return downloaded


downloaded_videos = download_videos(VIDEOS, VID_DIR)

print("Total downloaded videos:", len(downloaded_videos))

Downloaded: T9m3ACzBTA4.mp4


Downloaded: B-2E53Rxt4U.mp4


Downloaded: iNVvvhsjdvM.mp4


Downloaded: bmAUr22ap8c.mp4


Downloaded: y02mOQudT3E.mp4


Downloaded: c-djMIii9jQ.mp4


Downloaded: qGWjm5K6Ux8.mp4


Downloaded: 4AwYxLFcX74.mp4


Downloaded: hIsYsd4yX7o.mp4


Downloaded: dqQX3URGRCo.mp4
Total downloaded videos: 10


In [ ]:
CLIPS_DIR = VID_DIR / "clips"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

print("Clips folder:", CLIPS_DIR)

Clips folder: /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips


In [ ]:
def trim_video(src_path, dst_path, clip_seconds=8, target_width=1280):
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    scale_expr = f"scale={target_width}:-2"

    cmd = [
        "ffmpeg", "-y",
        "-t", str(clip_seconds),
        "-i", str(src_path),
        "-vf", scale_expr,
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "23",
        "-c:a", "aac",
        "-b:a", "128k",
        str(dst_path)
    ]

    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)


for video in downloaded_videos:
    output_clip = CLIPS_DIR / f"{video.stem}_clip.mp4"
    trim_video(video, output_clip, CLIP_SECONDS, TARGET_WIDTH)
    print("Created clip:", output_clip.name)

print("All clips saved to:", CLIPS_DIR)

Created clip: T9m3ACzBTA4_clip.mp4
Created clip: B-2E53Rxt4U_clip.mp4
Created clip: iNVvvhsjdvM_clip.mp4
Created clip: bmAUr22ap8c_clip.mp4
Created clip: y02mOQudT3E_clip.mp4
Created clip: c-djMIii9jQ_clip.mp4
Created clip: qGWjm5K6Ux8_clip.mp4
Created clip: 4AwYxLFcX74_clip.mp4
Created clip: hIsYsd4yX7o_clip.mp4
Created clip: dqQX3URGRCo_clip.mp4
All clips saved to: /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips


In [ ]:
def extract_frames(video_path, output_root, fps=10):
    stem = video_path.stem
    out_dir = output_root / stem
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-vf", f"fps={fps}",
        str(out_dir / f"{stem}_%04d.jpg")
    ]

    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)


for clip in CLIPS_DIR.glob("*.mp4"):
    extract_frames(clip, FRAME_DIR, FPS)
    print("Frames extracted from:", clip.name)

all_frames = sorted(glob.glob(str(FRAME_DIR / "**/*.jpg"), recursive=True))
print("Total frames:", len(all_frames))

Frames extracted from: T9m3ACzBTA4_clip.mp4
Frames extracted from: B-2E53Rxt4U_clip.mp4
Frames extracted from: iNVvvhsjdvM_clip.mp4
Frames extracted from: bmAUr22ap8c_clip.mp4
Frames extracted from: y02mOQudT3E_clip.mp4
Frames extracted from: c-djMIii9jQ_clip.mp4
Frames extracted from: qGWjm5K6Ux8_clip.mp4
Frames extracted from: 4AwYxLFcX74_clip.mp4
Frames extracted from: hIsYsd4yX7o_clip.mp4
Frames extracted from: dqQX3URGRCo_clip.mp4
Total frames: 799


In [ ]:
all_images = sorted(glob.glob(str(FRAME_DIR / "**/*.jpg"), recursive=True))

random.shuffle(all_images)

cut = int(len(all_images) * (1 - VAL_SPLIT))
train_images = all_images[:cut]
val_images = all_images[cut:]

print("Train images:", len(train_images))
print("Validation images:", len(val_images))

Train images: 639
Validation images: 160


In [ ]:
pretrained_model = YOLO("yolo11n.pt")

def write_yolo_label(result, image_path, label_dir):
    label_dir.mkdir(parents=True, exist_ok=True)

    lines = []

    for box in result.boxes:
        class_id = int(box.cls[0])

        # COCO class 0 = person
        if class_id != 0:
            continue

        x, y, w, h = box.xywhn[0].cpu().tolist()
        lines.append(f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    if lines:
        label_path = label_dir / f"{image_path.stem}.txt"
        label_path.write_text("\n".join(lines))


def copy_and_label_images(image_list, image_out_dir, label_out_dir):
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)

    results = pretrained_model.predict(
        source=image_list,
        imgsz=1280,
        conf=0.25,
        verbose=False
    )

    for img_path, result in zip(image_list, results):
        img_path = Path(img_path)

        copied_image = image_out_dir / img_path.name
        shutil.copy2(img_path, copied_image)

        write_yolo_label(result, copied_image, label_out_dir)


copy_and_label_images(train_images, TRAIN_IMG, TRAIN_LAB)
copy_and_label_images(val_images, VAL_IMG, VAL_LAB)

print("Auto-labeling completed.")

Auto-labeling completed.


In [ ]:
train_img_count = len(list(TRAIN_IMG.glob("*.jpg")))
val_img_count = len(list(VAL_IMG.glob("*.jpg")))

train_label_count = len(list(TRAIN_LAB.glob("*.txt")))
val_label_count = len(list(VAL_LAB.glob("*.txt")))

print("Train images:", train_img_count)
print("Train labels:", train_label_count)

print("Validation images:", val_img_count)
print("Validation labels:", val_label_count)

Train images: 639
Train labels: 566
Validation images: 160
Validation labels: 142


In [ ]:
yaml_text = f"""
path: {DATA_DIR}
train: images/train
val: images/val
nc: 1
names: ['player']
"""

yaml_path = DATA_DIR / "data.yaml"
yaml_path.write_text(yaml_text)

print(yaml_path.read_text())


path: /content/drive/My Drive/DS&AI/Projects/player_detection/data
train: images/train
val: images/val
nc: 1
names: ['player']



In [ ]:
model = YOLO("yolo11n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=80,
    imgsz=1280,
    batch=8,
    patience=20,
    project=str(EXPORTS),
    name="detect_train"
)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/DS&AI/Projects/player_detection/data/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detect_train, nbs=64, nms=False, opset=None, optimize=False, 

Additional code's for report bcz of crashing

In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = "player_detection"

BASE = Path("/content/drive/My Drive/DS&AI/Projects") / PROJECT
DATA_DIR = BASE / "data"
EXPORTS = BASE / "runs"

yaml_path = DATA_DIR / "data.yaml"
best_model_path = EXPORTS / "detect_train" / "weights" / "best.pt"

print("YAML exists:", yaml_path.exists())
print("Best model exists:", best_model_path.exists())

YAML exists: True
Best model exists: True


In [ ]:
trained_model = YOLO(str(best_model_path))

metrics = trained_model.val(
    data=str(yaml_path),
    imgsz=640,
    batch=4,
    workers=0
)

print(metrics.results_dict)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 9.5±19.8 ms, read: 13.0±13.2 MB/s, size: 59.5 KB)
val: Scanning /content/drive/My Drive/DS&AI/Projects/player_detection/data/labels/val.cache... 142 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 160/160 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 10.1it/s 4.0s
                   all        160       1016      0.825      0.729      0.839      0.623
Speed: 0.3ms preprocess, 5.0ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/runs/detect/val
{'metrics/precision(B)': 0.8245012992972959, 'metrics/recall(B)': 0.7293307086614174, 'metrics/mAP50(B)': 0.8392321606363546, 'metrics/mAP50-95(B)': 0.6227951796928965, 'fitness': 0.6227951796928965}


In [ ]:
precision = metrics.results_dict["metrics/precision(B)"]
recall = metrics.results_dict["metrics/recall(B)"]
map50 = metrics.results_dict["metrics/mAP50(B)"]
map50_95 = metrics.results_dict["metrics/mAP50-95(B)"]

print("Precision:", precision)
print("Recall:", recall)
print("mAP50:", map50)
print("mAP50-95:", map50_95)

Precision: 0.8245012992972959
Recall: 0.7293307086614174
mAP50: 0.8392321606363546
mAP50-95: 0.6227951796928965


In [ ]:
results_img = EXPORTS / "detect_train" / "results.png"
print("Results image exists:", results_img.exists())
print(results_img)

Results image exists: False
/content/drive/My Drive/DS&AI/Projects/player_detection/runs/detect_train/results.png


End***********************************************

In [ ]:
best_models = sorted(glob.glob(str(EXPORTS / "**/weights/best.pt"), recursive=True))

print("Best model files found:")
for model_path in best_models:
    print(model_path)

best_model_path = best_models[-1]

print("Using best model:", best_model_path)

Best model files found:
/content/drive/My Drive/DS&AI/Projects/player_detection/runs/detect_train/weights/best.pt
Using best model: /content/drive/My Drive/DS&AI/Projects/player_detection/runs/detect_train/weights/best.pt


In [ ]:
trained_model = YOLO(best_model_path)

metrics = trained_model.val(data=str(yaml_path))

print(metrics.results_dict)

Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 2.3±4.1 ms, read: 22.2±18.2 MB/s, size: 59.5 KB)
val: Scanning /content/drive/My Drive/DS&AI/Projects/player_detection/data/labels/val.cache... 142 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 160/160 23.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.1it/s 8.7s
                   all        160       1016      0.839       0.79      0.876      0.712
Speed: 16.3ms preprocess, 11.8ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /content/runs/detect/val-2
{'metrics/precision(B)': 0.8389936942662147, 'metrics/recall(B)': 0.7898469867484258, 'metrics/mAP50(B)': 0.8756231970518598, 'metrics/mAP50-95(B)': 0.7121180138267579, 'fitness': 0.7121180138267579}


For report Details

In [ ]:
precision = metrics.results_dict["metrics/precision(B)"]
recall = metrics.results_dict["metrics/recall(B)"]
map50 = metrics.results_dict["metrics/mAP50(B)"]
map50_95 = metrics.results_dict["metrics/mAP50-95(B)"]

print("Precision:", precision)
print("Recall:", recall)
print("mAP50:", map50)
print("mAP50-95:", map50_95)

Precision: 0.8389936942662147
Recall: 0.7898469867484258
mAP50: 0.8756231970518598
mAP50-95: 0.7121180138267579


End*****************************

In [ ]:
pred_dir = EXPORTS / "predictions"

trained_model.predict(
    source=str(VAL_IMG),
    imgsz=1280,
    conf=0.25,
    save=True,
    project=str(pred_dir),
    name="val_predictions"
)

print("Prediction images saved to:", pred_dir)


image 1/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0002.jpg: 736x1280 5 players, 67.9ms
image 2/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0007.jpg: 736x1280 2 players, 13.0ms
image 3/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0017.jpg: 736x1280 3 players, 13.0ms
image 4/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0020.jpg: 736x1280 5 players, 13.0ms
image 5/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0023.jpg: 736x1280 3 players, 12.9ms
image 6/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0040.jpg: 736x1280 5 players, 13.0ms
image 7/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0041.jpg: 736x1280 4 players, 13.0ms
image 8/160 /content/drive/My Dri

In [ ]:
CLIPS_DIR = BASE / "videos" / "clips"

print("Clips folder:", CLIPS_DIR)
print("Exists:", CLIPS_DIR.exists())

clips = sorted(CLIPS_DIR.glob("*.mp4"))
print("Number of clips:", len(clips))

for clip in clips:
    print(clip.name)

Clips folder: /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips
Exists: True
Number of clips: 10
4AwYxLFcX74_clip.mp4
B-2E53Rxt4U_clip.mp4
T9m3ACzBTA4_clip.mp4
bmAUr22ap8c_clip.mp4
c-djMIii9jQ_clip.mp4
dqQX3URGRCo_clip.mp4
hIsYsd4yX7o_clip.mp4
iNVvvhsjdvM_clip.mp4
qGWjm5K6Ux8_clip.mp4
y02mOQudT3E_clip.mp4


In [ ]:
track_dir = EXPORTS / "tracks"

for clip in sorted(CLIPS_DIR.glob("*.mp4")):
    print("Tracking:", clip.name)

    trained_model.track(
        source=str(clip),
        tracker="bytetrack.yaml",
        conf=0.25,
        iou=0.5,
        imgsz=1280,
        save=True,
        project=str(track_dir),
        name=clip.stem,
        verbose=False
    )

print("Tracking videos saved to:", track_dir)

Tracking: 4AwYxLFcX74_clip.mp4
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to /content/drive/My Drive/DS&AI/Projects/player_detection/runs/tracks/4AwYxLFcX74_clip
Tracking: B-2E53Rxt4U_clip.mp4
WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # gen

## Bonus: Keypoint Detection using YOLO Pose

This section implements a pose estimation model similar to OpenPose. It detects player body keypoints such as shoulders, elbows, wrists, hips, knees, and ankles.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

pose_model = YOLO("yolo11n-pose.pt")

print("Pose model loaded successfully.")

Pose model loaded successfully.


In [ ]:
pose_img_dir = EXPORTS / "pose_images"

pose_results = pose_model.predict(
    source=str(VAL_IMG),
    imgsz=640,
    conf=0.25,
    save=True,
    project=str(pose_img_dir),
    name="val_pose_predictions"
)

print("Pose image results saved to:", pose_img_dir)


image 1/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0002.jpg: 384x640 2 persons, 104.5ms
image 2/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0007.jpg: 384x640 2 persons, 42.9ms
image 3/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0017.jpg: 384x640 2 persons, 12.0ms
image 4/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0020.jpg: 384x640 2 persons, 11.6ms
image 5/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0023.jpg: 384x640 2 persons, 12.1ms
image 6/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0040.jpg: 384x640 4 persons, 11.4ms
image 7/160 /content/drive/My Drive/DS&AI/Projects/player_detection/data/images/val/4AwYxLFcX74_clip_0041.jpg: 384x640 5 persons, 11.8ms
image 8/160 /content/drive/My Drive/DS&

In [ ]:
pose_video_dir = EXPORTS / "pose_videos"

for clip in sorted(CLIPS_DIR.glob("*.mp4")):
    print("Running pose detection:", clip.name)

    pose_model.predict(
        source=str(clip),
        imgsz=640,
        conf=0.25,
        save=True,
        project=str(pose_video_dir),
        name=clip.stem
    )

print("Pose video results saved to:", pose_video_dir)

Running pose detection: 4AwYxLFcX74_clip.mp4

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/480) /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips/4AwYxLFcX74_clip.mp4: 384x640 1 person, 11.6ms
video 1/1 (frame 2/480) /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips/4AwYxLFcX74_clip.mp4: 384x640 1 person, 10.2ms
video 1/1 (frame 3/480) /content/drive/My Drive/DS&AI/Projects/player_detection/videos/clips/4AwYxLFcX74_clip.mp4: 384x640 1 person, 